# BI - LSTM MODEL Tonal Analysis For Voice-Based Sentiment Analysis

In [2]:
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader, random_split
from sklearn.metrics import accuracy_score, f1_score
from torch.nn.utils.rnn import pad_sequence

In [3]:
import os
import torch
import torchaudio
from torch.utils.data import Dataset
import torchaudio.transforms as T
import torchaudio.functional as F
import pandas as pd


In [62]:
# This is also Present in the dataset.py
class MFCCData(Dataset):
    def __init__(self, data, aud, label, max_len=300):
        self.data = pd.read_csv(data)
        self.audio_files = aud
        self.label_map = label
        self.sample_rate = 16000
        self.max_len = max_len
        self.delta = T.ComputeDeltas(win_length=5)
        self.delta_sq = T.ComputeDeltas(win_length=5)
        
        self.to_mfcc = T.MFCC(
            n_mfcc = 40,
            sample_rate = self.sample_rate,
            melkwargs={
                "n_fft": 400,
                "hop_length": 160,
                "n_mels": 64,
                "center": True,
                "power": 2.0
            }
        )
    
    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, index):
        row = self.data.iloc[index]
        wav = os.path.join(self.audio_files, f'dia{row['Dialogue_ID']}_utt{row['Utterance_ID']}' + '.wav')
        label = self.label_map[row['Emotion']]
        
        wave, sr = torchaudio.load(wav)
        mfcc = self.to_mfcc(wave)
        delta = self.delta(mfcc)
        delta_sq = self.delta_sq(delta)
        e_mfcc = torch.cat([mfcc, delta, delta_sq], dim=1).squeeze(0).transpose(0,1)
              
        T_current, D = e_mfcc.shape
        if T_current < self.max_len:
            pad = torch.zeros(self.max_len - T_current, D)
            e_mfcc = torch.cat([e_mfcc, pad], dim=0)
        else:
            e_mfcc = e_mfcc[:self.max_len, :]
        
        return e_mfcc, torch.tensor(label)

In [80]:
class Attention(nn.Module):
    def __init__(self, hidden):
        super(Attention, self).__init__()
        self.attn = nn.Linear(hidden * 2, 1)
        
    def forward(self, ltsm_out):
        weights = torch.softmax(self.attn(ltsm_out), dim=1)
        weighted = torch.sum(ltsm_out * weights, dim=1)
        return weighted        

In [89]:
# Available in models.py
class BiLSTM(nn.Module):
    def __init__(self, input_dim = 120, hidden_dim = 128, num_layers=2, output_dim=3, dropout=0.3):
        super(BiLSTM, self).__init__()
        self.lstm = nn.LSTM(
            input_size=input_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            bidirectional=True,
            dropout=dropout
        )
        
        self.fc = nn.Linear(hidden_dim * 2, output_dim)
        self.dropout = nn.Dropout(dropout)
        self.attn = Attention(hidden_dim)
        self.classifier = nn.Sequential(
                            nn.Linear(hidden_dim * 2, hidden_dim),
                            nn.ReLU(),
                            nn.Dropout(dropout),
                            nn.Linear(hidden_dim, output_dim)
                        )
        
    def forward(self, x):
        lstm_out, _ = self.lstm(x)
        pool = self.attn(lstm_out)
        out = self.classifier(pool)
        return out

In [90]:
def train(device, model, dataloader, criterion, optimizer):
    model.train()
    total_loss = 0
    all_preds, all_labels = [], []
    
    for mfcc, labels in dataloader:
        mfcc, labels = mfcc.to(device), labels.to(device)
        
        optimizer.zero_grad()
        output = model(mfcc)
        loss = criterion(output, labels)
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
        preds = torch.argmax(output, dim=1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
    
    acc = accuracy_score(all_labels, all_preds)
    f1 = f1_score(all_labels, all_preds, average='macro')
    avg_loss = total_loss / len(dataloader)
    return avg_loss, f1, acc
    

In [91]:
def eval(model, dataloader, criterion, device):
    model.eval()
    total_loss = 0
    all_preds, all_labels = [], []
    
    with torch.no_grad():
        for mfcc, label in dataloader:
            mfcc, label = mfcc.to(device), label.to(device)
            output = model(mfcc)
            loss = criterion(output, label)
            total_loss += loss
            
            preds = torch.argmax(output, dim=1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(label.cpu().numpy())
            
    acc = accuracy_score(all_labels, all_preds)
    f1 = f1_score(all_labels, all_preds, average='macro')
    avg_loss = total_loss / len(dataloader)
    return avg_loss, f1, acc

In [92]:
labels_map = {
    "neutral": 0,
    'joy': 1,
    "surprise": 1,
    'sadness': 2,
    'anger': 2,
    'disgust': 2,
    'fear': 2
}
dataset = MFCCData(data='MELD.Raw/train_sent_emo.csv', aud='MELD.Wav/train/', label=labels_map)

In [93]:
# Set your split lengths
total_size = len(dataset)
train_size = int(0.8 * total_size)
val_size = total_size - train_size

training, validation = random_split(dataset, [train_size, val_size])


In [94]:
def collate(batch):
    from torch.nn.utils.rnn import pad_sequence
    mfccs, labels = zip(*batch)
    
    # No need to re-wrap if they're already tensors
    padded_mfccs = pad_sequence(mfccs, batch_first=True)  # Shape: (batch, time, features)
    labels = torch.tensor(labels)
    return padded_mfccs, labels

In [95]:
train_loader = DataLoader(training, batch_size=32, shuffle=True, collate_fn=collate)
val_loader = DataLoader(validation, batch_size=32, shuffle=False, collate_fn=collate)


In [96]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = BiLSTM().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=2, verbose=True)
criterion = nn.CrossEntropyLoss()

best_val_f1 = 0
patience = 5
patience_counter = 0

num_epochs = 100

for epoch in range(num_epochs):
    train_loss, train_acc, train_f1 = train(model=model, dataloader=train_loader, optimizer=optimizer, criterion=criterion, device=device)
    val_loss, val_acc, val_f1 = eval(model=model, dataloader=val_loader, criterion=criterion, device=device)
    
    scheduler.step(val_f1)

    print(f"Epoch {epoch+1}/{num_epochs}")
    print(f"Train Loss: {train_loss:.4f} | Acc: {train_acc:.4f} | F1: {train_f1:.4f}")
    print(f"Val   Loss: {val_loss:.4f} | Acc: {val_acc:.4f} | F1: {val_f1:.4f}")
    print("-" * 50)

    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        patience_counter = 0
        torch.save(model.state_dict(), "best_bilstm_model.pt")
        print("🔥 Best model saved!")
    else:
        patience_counter += 1
        if patience_counter >= patience:
            print("⏹️ Early stopping triggered.")
            break

/home/kurty/Project/lib/python3.12/site-packages/torch/optim/lr_scheduler.py:62: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn(


Epoch 1/100
Train Loss: 1.0368 | Acc: 0.2899 | F1: 0.4814
Val   Loss: 1.0250 | Acc: 0.3102 | F1: 0.4920
--------------------------------------------------
🔥 Best model saved!
Epoch 2/100
Train Loss: 1.0185 | Acc: 0.3601 | F1: 0.5008
Val   Loss: 1.0219 | Acc: 0.3714 | F1: 0.4975
--------------------------------------------------
🔥 Best model saved!
Epoch 3/100
Train Loss: 1.0016 | Acc: 0.4067 | F1: 0.5186
Val   Loss: 1.0386 | Acc: 0.3795 | F1: 0.4935
--------------------------------------------------
Epoch 4/100
Train Loss: 0.9825 | Acc: 0.4287 | F1: 0.5290
Val   Loss: 1.0247 | Acc: 0.4136 | F1: 0.4920
--------------------------------------------------
Epoch 5/100
Train Loss: 0.9569 | Acc: 0.4730 | F1: 0.5591
Val   Loss: 1.0460 | Acc: 0.4106 | F1: 0.4905
--------------------------------------------------
Epoch 6/100
Train Loss: 0.8786 | Acc: 0.5480 | F1: 0.6069
Val   Loss: 1.0584 | Acc: 0.4226 | F1: 0.4870
--------------------------------------------------
Epoch 7/100
Train Loss: 0.7915

In [ ]:
# Created by DaBloat